In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.expand_frame_repr', False)

In [31]:
train = pd.read_csv('../data/raw/train.csv', low_memory=False, parse_dates=['Date'])
store = pd.read_csv('../data/raw/store.csv')
data = pd.merge(train, store, on='Store', how='left')

In [7]:
total = len(data)
closed = (data['Open'] == 0).sum()
open_no_sales = ((data['Open'] == 1) & (data['Sales'] == 0)).sum()
open_sales = ((data['Open'] == 1) & (data['Sales'] > 0)).sum()

print(f'Total records: {total}')
print(f'Closed: {closed}, {closed/total:.2%}')
print(f'Open with no sales: {open_no_sales}, {open_no_sales/total:.2%}')
print(f'Open with sales: {open_sales}, {open_sales/total:.2%}')

Total records: 1017209
Closed: 172817, 16.99%
Open with no sales: 54, 0.01%
Open with sales: 844338, 83.01%


> Almost 17% rows represent Sundays, Holidays and Occasional Closures. The Sales are mechanically 0.

> Around 83% rows showcase normal business days.

> A very minimal part of the dataset, around 0.01% showcase abnormal behaviour, open store but no sales. These must be data-entry gaps or system outages. We'll investigate it before dropping it.

In [12]:
open_sale_zero = data[(data['Open'] == 1) & (data['Sales'] == 0)]

print('------------------Rows where stores are open but sales are zero----------------------')
print(f'Total such rows: {len(open_sale_zero)}')
print('No. of unique stores: ', open_sale_zero['Store'].nunique())
print('\nTop stores by count: ')
print(open_sale_zero['Store'].value_counts().head(10))
print('Date Range: ', open_sale_zero['Date'].min(), 'to', open_sale_zero['Date'].max())
print('\nNational Holidays: ')
print(open_sale_zero[['StateHoliday', 'SchoolHoliday']].value_counts())


------------------Rows where stores are open but sales are zero----------------------
Total such rows: 54
No. of unique stores:  41

Top stores by count: 
Store
28      3
835     2
102     2
1017    2
1100    2
25      2
623     2
983     2
1039    2
665     2
Name: count, dtype: int64
Date Range:  2013-01-17 to 2015-05-15

National Holidays: 
StateHoliday  SchoolHoliday
0             0                42
              1                12
Name: count, dtype: int64


> Total 54 records show 0 sales even when the store was open. These records are spread across 41 stores, suggesting it is just a random anamoly.

> Store 28 has 3 such records and others have 2 or 1 such records. Thus, such minimal amount of rows can be dropped without affecting the results.

> Other Checks:
- Date ranges almost across the whole time span of dataset.
- 42 such records were on a normal business day (no national holiday)

In [13]:
data_open = data[(data['Open'] == 1) & (data['Sales'] > 0)].reset_index(drop=True)

print('Before filter length: ', len(data))
print('After filter length: ', len(data_open))
print(f'Removed {(len(data) - len(data_open))} rows where stores were open but had zero sales, which is {(len(data) - len(data_open))/len(data):.2%} of the data.')

Before filter length:  1017209
After filter length:  844338
Removed 172871 rows where stores were open but had zero sales, which is 16.99% of the data.


This 'After filter length' 844338 matches the number we expected above. 

In [16]:
# Null calculation on the filtered data
print('Null Data on Open with Sales records: ')
null = data_open.isnull().sum()
print(null[null > 0])

print("\n% of null values in Open with Sales records: \n", (null[null > 0] / len(data_open) * 100).round(2))

Null Data on Open with Sales records: 
CompetitionDistance            2186
CompetitionOpenSinceMonth    268600
CompetitionOpenSinceYear     268600
Promo2SinceWeek              423292
Promo2SinceYear              423292
PromoInterval                423292
dtype: int64

% of null values in Open with Sales records: 
 CompetitionDistance           0.26
CompetitionOpenSinceMonth    31.81
CompetitionOpenSinceYear     31.81
Promo2SinceWeek              50.13
Promo2SinceYear              50.13
PromoInterval                50.13
dtype: float64


- Competition Distance nulls form 0.26% of the filtered data. These are the records that simply didn't capture the data but it exists. So, we'll fill the nulls with column median. As it's just a small fraction of data, it won't affect the aggregate as much.

- Promo2SinceWeek, Promo2SinceYear, PromoInterval are missing always as a unit (all 50.13%). This tells us that these records just didn't participate in Promo2. So we fill these with a placeholder 'NA'.

- CompetitionOpenSinceMonth and CompetitionOpenSinceYear are genuine nulls that aren't recoverable. We cannot guess the opening date. So, we fill this with a placeholder and flag it with missing-flag column.

In [17]:
data_open['PromoInterval'] = data_open['PromoInterval'].fillna('None')
data_open['Promo2SinceWeek'] = data_open['Promo2SinceWeek'].fillna(0)
data_open['Promo2SinceYear'] = data_open['Promo2SinceYear'].fillna(0)

print(data_open[['PromoInterval', 'Promo2SinceWeek', 'Promo2SinceYear']].isnull().sum())

PromoInterval      0
Promo2SinceWeek    0
Promo2SinceYear    0
dtype: int64


In [18]:
med = data_open['CompetitionDistance'].median()
print('Median Competition Distance in meters (from non-null rows): ', med, 'm')

data_open['CompetitionDistance'] = data_open['CompetitionDistance'].fillna(med)
print('Nulls Remaining: ', data_open['CompetitionDistance'].isnull().sum())

Median Competition Distance in meters (from non-null rows):  2320.0 m
Nulls Remaining:  0


- The median competition distance is 2320 meters. We use median instead of mean as the distance is usually right skewed; mostly the competition is a few hundred meters away in urban towns. So mean value is affected by some outlier cases thus we consider median value.

- We filled 0.26% of the missing values with the median of 2320.

In [19]:
data_open['CompetitionInfoMissing'] = data_open['CompetitionOpenSinceYear'].isnull().astype(int)

data_open['CompetitionOpenSinceMonth'] = data_open['CompetitionOpenSinceMonth'].fillna(0)
data_open['CompetitionOpenSinceYear'] = data_open['CompetitionOpenSinceYear'].fillna(0)

print('Nulls Remaining: ')
print(data_open[['CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear']].isnull().sum())
print('\nFlag Distribution: (1 = Info was missing)')
print(data_open['CompetitionInfoMissing'].value_counts())

Nulls Remaining: 
CompetitionOpenSinceMonth    0
CompetitionOpenSinceYear     0
dtype: int64

Flag Distribution: (1 = Info was missing)
CompetitionInfoMissing
0    575738
1    268600
Name: count, dtype: int64


- We used missing-indicator technique here. The missing info could be useful here meaning the competition either doesn't exist or the info is unknown. We make a new binary column where 1 means that info was missing and 0 means it was present. Then we fill the missing rows with 0 - an impossible date. Thus, we resolved the missing rows while maintaining the missing info.

In [21]:
# Here, the null-handling section ends so we verify that all nulls have been handled
print('Final null check: ')
final_null = data_open.isnull().sum()
print(final_null[final_null > 0] if final_null.sum() > 0 else 'No nulls')

Final null check: 
No nulls


In [22]:
print('Current dtype: ', data_open['StateHoliday'].dtype)
print('\nWhat values actually exist and their Python types: ')
for v in data_open['StateHoliday'].unique():
    print(f'{repr(v)} -> {type(v).__name__}')

Current dtype:  str

What values actually exist and their Python types: 
'0' -> str
'a' -> str
'b' -> str
'c' -> str


- EDA's initial read raised a mixed-type warning on StateHoliday, caused by pandas' chunked dtype inference. Resolved by reading with low_memory=False, forcing single-pass inference; column is uniformly string-typed ('0','a','b','c').

In [23]:
cat_cols = ['StoreType', 'Assortment', 'StateHoliday', 'PromoInterval']
for c in cat_cols:
    print(f'{c}: {data_open[c].nunique()} unique -> {sorted(data_open[c].unique())}')
    

StoreType: 4 unique -> ['a', 'b', 'c', 'd']
Assortment: 3 unique -> ['a', 'b', 'c']
StateHoliday: 4 unique -> ['0', 'a', 'b', 'c']
PromoInterval: 4 unique -> ['Feb,May,Aug,Nov', 'Jan,Apr,Jul,Oct', 'Mar,Jun,Sept,Dec', 'None']


- We convert these categorical values into fixed encoded labels to save memory and clear the intent of these columns being fixed finite labels.

In [24]:
# Memory Measure
before = data_open.memory_usage(deep=True).sum() / 1024**2
print(f'Memory Before: {before:.1f} MB')

cat_cols = ['StoreType', 'Assortment', 'StateHoliday', 'PromoInterval']
for c in cat_cols:
    data_open[c] = data_open[c].astype('category')

after = data_open.memory_usage(deep=True).sum() / 1024**2
print(f'Memory After: {after:.1f} MB')
print(f'Memory Reduction: {before - after:.1f} MB ({(before - after)/before:.0%})')

print('\nData Types after conversion: ')
print(data_open[cat_cols].dtypes)

Memory Before: 305.6 MB
Memory After: 140.9 MB
Memory Reduction: 164.7 MB (54%)

Data Types after conversion: 
StoreType        category
Assortment       category
StateHoliday     category
PromoInterval    category
dtype: object


- Thus by converting the dtypes to category, we saved 164.7 MB of memory and reduced the overall memory usage by 54%

In [25]:
# Checking for float columns that might be better as integers
int_candidates = ['CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2SinceWeek', 'Promo2SinceYear']
print(data_open[int_candidates].dtypes)

CompetitionOpenSinceMonth    float64
CompetitionOpenSinceYear     float64
Promo2SinceWeek              float64
Promo2SinceYear              float64
dtype: object


CompetitionOpenSinceMonth/Year and Promo2SinceWeek/Year were forced to be float columns before, because NaN itself is a float value, so any numeric column containing even one null must be float. 

Now that we've filled every null, that constraint is gone. A year is conceptually a whole number; 2013.0 is just visual noise that'll leak into our tables and plots. Converting to int is not correctness-critical but it makes the frame tidy.

In [26]:
for c in int_candidates:
    data_open[c] = data_open[c].astype(int)
print(data_open[int_candidates].dtypes)

CompetitionOpenSinceMonth    int64
CompetitionOpenSinceYear     int64
Promo2SinceWeek              int64
Promo2SinceYear              int64
dtype: object


In [29]:
print('Date Type: ', data_open['Date'].dtype)
print('First value: ', repr(data_open['Date'].iloc[0]))
print("Type of first value:", type(data_open['Date'].iloc[0]).__name__)

Date Type:  str
First value:  '2015-07-31'
Type of first value: str


In [30]:
data_open['Date'] = pd.to_datetime(data_open['Date'])
print(data_open['Date'].dtype)

datetime64[us]


In [41]:
# Calendar features
data_open['Year'] = data_open['Date'].dt.year
data_open['Month'] = data_open['Date'].dt.month
data_open['Day'] = data_open['Date'].dt.day
data_open['WeekOfYear'] = data_open['Date'].dt.isocalendar().week.astype(int)

print(data_open[['Date', 'Year', 'Month', 'Day', 'WeekOfYear', 'DayOfWeek']].head())
print('Year range: ', data_open['Year'].min(), 'to', data_open['Year'].max())
print('Month range: ', data_open['Month'].min(), 'to', data_open['Month'].max())
print('Day range: ', data_open['Day'].min(), 'to', data_open['Day'].max())
print('WeekOfYear range: ', data_open['weekOfYear'].min(), 'to', data_open['weekOfYear'].max())

        Date  Year  Month  Day  WeekOfYear  DayOfWeek
0 2015-07-31  2015      7   31          31          5
1 2015-07-31  2015      7   31          31          5
2 2015-07-31  2015      7   31          31          5
3 2015-07-31  2015      7   31          31          5
4 2015-07-31  2015      7   31          31          5
Year range:  2013 to 2015
Month range:  1 to 12
Day range:  1 to 31
WeekOfYear range:  1 to 52


In [44]:
data_open = data_open.drop(columns=['weekOfYear'])

We just spent several steps fixing dtypes. If we save to CSV, every one of those is thrown away as CSV has no concept of dtype. Parquet stores the schema with the data, so when we re-load it, Date is still datetime, categoricals are still categories and ints are still ints. The work persists.

In [45]:
import os
os.makedirs('../data/processed', exist_ok=True)

out_path = '../data/processed/rossmann_clean.parquet'
data_open.to_parquet(out_path, index=False, engine='fastparquet')

# verify the round-trip: read it back and confirm types survived
check = pd.read_parquet(out_path, engine='fastparquet')
print(f"Saved {len(check):,} rows, {check.shape[1]} columns")
print("\nDtypes after round-trip (the proof Parquet preserved them):")
print(check.dtypes)
print("\nNulls after round-trip:", check.isnull().sum().sum())

Saved 844,338 rows, 23 columns

Dtypes after round-trip (the proof Parquet preserved them):
Store                                 int64
DayOfWeek                             int64
Date                         datetime64[us]
Sales                                 int64
Customers                             int64
Open                                  int64
Promo                                 int64
StateHoliday                       category
SchoolHoliday                         int64
StoreType                          category
Assortment                         category
CompetitionDistance                 float64
CompetitionOpenSinceMonth             int64
CompetitionOpenSinceYear              int64
Promo2                                int64
Promo2SinceWeek                       int64
Promo2SinceYear                       int64
PromoInterval                      category
CompetitionInfoMissing                int64
Year                                  int32
Month                       